# Занятие 5, пара 1. Предобработка. Аномалии, пропуски, типы

Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai

## Данные получили. Теперь можно им верить?

Данные редко бывают чистыми и красивыми. В прошлом блоке мы учились получать записи; теперь будем разбираться, что в них означает каждое число. Если средний возраст посетителей курса получился 130 лет, это не обязательно необыкновенно опытная аудитория. Возможно, стоит сначала посмотреть на таблицу.

После пары вы сможете отличить пропуск от нуля, проверить тип и единицу измерения, заметить подозрительное значение и объяснить решение. **Удалить всё некрасивое — не наша цель.** Сохраним исходник и будем менять только копию.

Предполагается, что вы знакомы с DataFrame, выбором столбцов, масками и `.loc` из блока 02, а также со строковыми методами из блока 03. Полностью проходить API до этой тетради не требуется: исходные данные здесь уже в локальном CSV.

### Подготовка

Основные примеры работают без «Ваш ход». Пропускаете задание — пропустите и его `check(...)`, затем продолжайте обычный раздел. Решения упражнений не нужны для демонстрации. Сохраняйте свои ответы в отдельной копии.

Нужны Pandas и Matplotlib, сеть не требуется. Служебная ячейка ниже подключает `course_support` у текущей папки или её родителей. `DATA_DIR` — учебные исходники, `WORKSPACE_DIR` — ваша рабочая папка. Эту настройку не нужно писать по памяти.

In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Что лежит в нашем файле

Это вымышленные заявки на короткие курсы: одна строка — запись выгрузки. Из-за повторов запись ещё не обязательно означает новую заявку. У нас нет реальных персональных данных и нет задачи делать выводы о настоящем учебном центре.

`pd.read_csv` знаком по «Титанику». Теперь явно зададим `sep=';'`: поля разделены точкой с запятой. `dtype='string'` сначала оставит **все поля строками**, в том числе ID `001`. Автоматическое превращение ID в число уничтожило бы ведущие нули. Числовые поля преобразуем отдельно, когда поймём их смысл.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

data_path = DATA_DIR / 'preprocessing' / 'requests_raw.csv'
raw = pd.read_csv(data_path, sep=';', dtype='string', encoding='utf-8')
display(raw.head())
print(raw.shape)
print(raw.columns)

Что успели заметить? У кого-то возраст 240, у кого-то стоимость 0. Оба числа необычные, но пока нельзя объявить оба ошибками. Откроем описание источника. Правила исходника важнее нашей догадки о том, как «обычно бывает».

In [ ]:
source_path = DATA_DIR / 'preprocessing' / 'SOURCE.md'
print(source_path.read_text(encoding='utf-8'))

### 2. Названия столбцов — договоримся один раз

В исходнике пробелы и заглавные буквы. Это допустимо, но при постоянном обращении легко забыть написание. Приведём заголовки к одному стилю. `rename(columns=словарь)` создаёт таблицу с переименованными столбцами: слева старое имя, справа новое. `.copy()` оставит нам отдельную рабочую таблицу.

Текстовые числовые поля назовём `age_raw`, `duration_raw`, `fee_raw`: дальше рядом появятся проверенные числовые версии. Сырые значения не теряем. Переименование не «чистит» данные автоматически, оно делает работу понятнее.

In [ ]:
names = {
    'Request ID': 'request_id',
    'Age': 'age_raw',
    'Education': 'education',
    'Duration': 'duration_raw',
    'Unit': 'unit',
    'Fee RUB': 'fee_raw',
    'Format': 'format'
}
renamed = raw.rename(columns=names)
df = renamed.copy()
df.info()

### 3. Похоже на число — ещё не число

Если `'19'` хранится текстом, арифметика не получит возраст сама по себе. `pd.to_numeric` преобразует Series в числа. Параметр `errors='coerce'` превращает **неразобранное значение** в пропуск. Удобно? Да. Опасно? Тоже: без отдельного просмотра мы потеряем причину.

Сначала проверим маленький пример, где заранее знаем ответы. `isna()` уже знаком: отмечает пропуски. `notna()` делает обратное. Условие «было заполнено, но не стало числом» выделит именно проблему преобразования.

In [ ]:
number_text = pd.Series(['19', 'неизвестно', None], dtype='string')
numbers = pd.to_numeric(number_text, errors='coerce')
display(numbers)
conversion_failed = number_text.notna() & numbers.isna()
display(number_text.loc[conversion_failed])

Вторая строка не разобрана, третья была пустой с самого начала. Это разные причины одного внешнего результата — пропуска.

Преобразуем возраст. `describe()` показывает число известных значений и основные характеристики; эта сводка помогает заметить подозрительное, но сама не объясняет причину.

In [ ]:
df['age'] = pd.to_numeric(df['age_raw'], errors='coerce')
display(df[['request_id', 'age_raw', 'age']])
display(df['age'].describe())

По правилам **этой учебной формы** допустимы целые 16…100. Это ограничение конкретного источника, не определение предельного человеческого возраста. Создадим две маски и объединим знакомым `|` — «или». Скобки отделяют каждое сравнение.

Для неизвестного возраста сравнение может дать `<NA>`, а не True/False. `.fillna(False)` у маски означает «не считаем неизвестное значение подтверждённой ошибкой диапазона». Сам возраст этим действием не заполняется.

In [ ]:
bad_age = (df['age'] < 16) | (df['age'] > 100)
df['age_invalid'] = bad_age.fillna(False)
display(df.loc[df['age_invalid'], ['request_id', 'age_raw', 'age']])

240 точно нарушает правило, но правильное число неизвестно. Исправить на 24 «потому что похоже» было бы выдумыванием данных. В числовой версии оставим `pd.NA` — явный пропуск; исходный текст и признак ошибки сохранятся.

Тип Pandas `'Int64'` с большой I хранит целые значения вместе с пропуском. Обычное преобразование к `int` не решает задачу неизвестного возраста.

In [ ]:
df.loc[bad_age, 'age'] = pd.NA
df['age'] = df['age'].astype('Int64')
display(df[['request_id', 'age_raw', 'age', 'age_invalid']])

### 4. Три часа и 180 минут — не разные масштабы занятий

Преобразуем длительность в число и сохраним новую колонку `hours`. Сначала она копирует числовую длительность. Затем только у записей с `unit == 'мин'` разделим её на 60. `.loc[маска, столбец]` позволяет изменить нужные строки.

Единицы берём из столбца, а не угадываем по величине. Число 180 могло бы означать долгий курс, но источник прямо говорит «минуты». Здесь час астрономический — 60 минут, не академический.

In [ ]:
df['duration'] = pd.to_numeric(df['duration_raw'], errors='coerce')
df['hours'] = df['duration'].copy()
minutes = df['unit'] == 'мин'
df.loc[minutes, 'hours'] = df.loc[minutes, 'duration'] / 60
display(df[['request_id', 'duration_raw', 'unit', 'hours']])

А отрицательные четыре часа? Ошибка действительно есть, но источник не подтвердил, что это просто неверный знак. `abs()` сделало бы число приятнее, не доказав его истинность. Отметим проблему и оставим неизвестность. Не удаляем всю заявку: другие поля ещё полезны.

In [ ]:
df['hours_invalid'] = (df['hours'] < 0).fillna(False)
df.loc[df['hours_invalid'], 'hours'] = pd.NA
display(df.loc[df['hours_invalid'], ['request_id', 'duration_raw', 'hours']])

### ✏️ Ваш ход — сопоставить единицы

Создайте небольшую таблицу `my_times` с длительностями `[90, 2, 150]` и единицами `['мин', 'час', 'мин']`. Добавьте `hours`, переводя только минуты в часы. Ожидаемые значения сначала посчитайте без Python. Затем объясните, почему нельзя делить весь столбец на 60.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('05.1.1', my_times['hours'])

**Мой вывод:** что изменили, на каком основании и что осталось неизвестным. …

### 5. Стоимость: сначала пробелы, потом числа

`'3 600'` не разберётся как число с пробелом внутри. В прошлый раз убирали пробелы у одной строки; теперь используем `.str` — доступ к строковым операциям для каждого значения Series. `.str.replace(' ', '', regex=False)` заменяет обычный пробел пустой строкой. `regex=False` означает буквальную замену, без регулярных выражений.

Мы знаем формат именно этого файла. Произвольные «3–5 тысяч» так чистить нельзя: склеится другое число. Отдельные шаги помогают заметить такую ошибку.

In [ ]:
fee_text = df['fee_raw'].str.replace(' ', '', regex=False)
display(fee_text)
df['fee_rub'] = pd.to_numeric(fee_text, errors='coerce')
fee_conversion_failed = df['fee_raw'].notna() & df['fee_rub'].isna()
display(df.loc[fee_conversion_failed, ['request_id', 'fee_raw', 'fee_rub']])

«Уточняется» стало пропуском, а ноль остался нулём. Для 004 источник подтвердил промоместо. Заменить этот ноль медианой — значит придумать плату за бесплатное место. Так что правило «нулей не бывает» здесь уже сломалось.

### ✏️ Ваш ход — не потерять ноль

В `my_fee_text` запишите Series `['1 200', '0', 'уточняется', None]` с `dtype='string'`. Получите `my_fee` числами и `my_failed` — маску случаев, когда исходное непустое значение не удалось разобрать. Не заменяйте неизвестность нулём.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('05.1.2', my_fee, my_failed)

**Мой вывод:** что изменили, на каком основании и что осталось неизвестным. …

### 6. Пропуски: сначала посчитаем, затем решим

`isna().sum()` знаком по «Титанику». Применим действия по отдельности к новым числовым полям. Сколько сведений отсутствовало изначально, а сколько мы признали недостоверными? Для ответа рядом остались исходные столбцы и признаки ошибок.

In [ ]:
numeric_fields = df[['age', 'hours', 'fee_rub']]
missing_mask = numeric_fields.isna()
missing_counts = missing_mask.sum()
display(missing_counts)
df['fee_missing'] = df['fee_rub'].isna()
display(df.loc[df['fee_missing'], ['request_id', 'fee_raw', 'fee_rub']])

Удалим все строки с пропусками? Можно, но сначала оценим потерю. `dropna(subset=[...])` возвращает строки с известными значениями перечисленных столбцов. В отдельной `complete` посмотрим последствия; рабочую `df` пока не заменяем.

Одинаковое число пропусков в двух столбцах ещё не доказывает, что отсутствуют одни и те же строки. Причину нельзя восстановить только по счётчику.

In [ ]:
complete = df.dropna(subset=['age', 'hours', 'fee_rub'])
print('Записей до:', len(df))
print('Полных по трём полям:', len(complete))
print('Потеряли бы:', len(df) - len(complete))

Число потерянных строк само по себе не разрешает и не запрещает удаление. Важен вопрос анализа: для длительности могут пригодиться записи без цены; для распределения цены — без возраста. Универсального правила «можно удалить не больше 5%» нет.

Другая идея — заполнить пропуски. `.fillna(значение)` вернёт Series с заменой пропусков. Медиана `.median()` считает середину известных значений. Попробуем на маленьком наборе, чтобы точно понимать последствия, **не меняя наш источник**.

In [ ]:
demo_fees = pd.Series([1000, 2000, 3000, None])
median_fee = demo_fees.median()
filled_fees = demo_fees.fillna(median_fee)
print('Медиана известных:', median_fee)
display(filled_fees)

Последнее значение теперь похоже на наблюдение, но на самом деле мы его придумали по правилу. Заполнение может быть полезно алгоритму, который не принимает пропуски, однако это не восстановление истины. Оно меняет распределение и уменьшает видимый разброс.

В нашем учебном наборе пока **оставим неизвестные стоимости неизвестными** и сохраним признак `fee_missing`. Для будущего ML параметры заполнения будут вычисляться только по train, иначе информация из проверки попадёт в обучение. Подробно разберём это в курсе ML, а сейчас не будем выдавать догадку за факт.

### ✏️ Ваш ход — увидеть цену заполнения

Создайте `my_values = pd.Series([1000, 2000, 9000, None])`. Получите `my_median`, `my_filled`, `my_missing_before`. Сравните среднее известных значений до заполнения со средним после. В объяснении укажите, какое число добавил метод и почему оно не является найденным настоящим значением.

In [ ]:
# ✏️ ваш код здесь

In [ ]:
check('05.1.3', my_median, my_missing_before, my_filled)

**Мой вывод:** что изменили, на каком основании и что осталось неизвестным. …

### 7. Одна строка — одно небольшое расследование

**Самостоятельно, 10–12 минут.** Выберите ID 003, 004, 008 или 012. Покажите исходные и новые поля через `.loc`. Напишите три короткие фразы: что известно из источника; что изменили кодом; что всё ещё неизвестно. Предложите вопрос владельцу источника, который реально помог бы выбрать исправление. Для 012 не требуйте ошибку только потому, что число большое.

Обсудите результат с соседом: какой шаг у вас подтверждён, а какой остался гипотезой? Эта работа не создаёт переменных для следующего раздела.

### 8. Сохраним промежуточный результат

Обработка ещё не закончена: повторы и написания образования остались на следующую пару. Сохраним состояние отдельно от исходника. `mkdir(parents=True, exist_ok=True)` создаёт папку при необходимости, `to_csv(index=False)` не записывает служебный индекс в отдельный столбец.

В комплекте есть такая же предоставленная промежуточная таблица для 05.2. Поэтому следующую тетрадь можно начать с чистого ядра, даже если ваш файл не оказался под рукой. Это не замена вашим ответам.

In [ ]:
output_dir = WORKSPACE_DIR / 'reports' / 'preprocessing'
output_dir.mkdir(parents=True, exist_ok=True)
stage_path = output_dir / 'requests_stage1.csv'
df.to_csv(stage_path, index=False, encoding='utf-8')
print(stage_path)
print('Сохранено записей:', len(df))

Итог этой пары — не «таблица без пропусков», а таблица с понятными преобразованиями и честно сохранённой неизвестностью. Проверьте себя: почему 0, пустое значение и неподходящий текст нельзя исправлять одним действием? Почему из 180 получили 3, а из −4 не получили 4?

Для чтения по желанию — [Pandas: работа с пропусками](https://pandas.pydata.org/docs/user_guide/missing_data.html) и раздел о пропусках в [главе 7 книги Уэса Маккинни](https://wesmckinney.com/book/data-cleaning.html). Не нужно читать всю главу перед следующей парой.